# 07 – Ferdig modell: Llama Prompt Guard 2

[`meta-llama/Llama-Prompt-Guard-2-86M`](https://huggingface.co/meta-llama/Llama-Prompt-Guard-2-86M) er Metas ferdigtrente klassifiserer for prompt-angrep. Den er bygd på `microsoft/mdeberta-v3-base`, samme grunnmodell som vår egen mDeBERTa, og er derfor flerspråklig. (22M-versjonen er bare på engelsk og er ikke med her.)

Spørsmålet er om en ferdig flerspråklig modell fra Meta er like god som den vi har trent selv, særlig på norsk.

**Forbehold:** Vi vet ikke nøyaktig hvilke data Meta trente på. Noen av de offentlige testsettene våre kan ha vært med i treningen deres. De egne testsettene har ingen slik lekkasje.

Modellen krever at du har fått tilgang på Hugging Face og er logget inn (`hf auth login`). Første kjøring laster ned modellen (ca. 1 GB) og tar 20–40 minutter på CPU. Scorene lagres i `models/promptguard_scores.parquet`, og sammenligningen med de andre modellene ligger i notebook 06.

In [1]:
import hashlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_calibration_no, load_own_testsets, load_training_data
from evaluate import evaluate_by_source, threshold_for_fpr

MODEL_NAME = "meta-llama/Llama-Prompt-Guard-2-86M"
CACHE = ROOT / "models" / "promptguard_scores.parquet"
torch.set_num_threads(max(1, torch.get_num_threads()))

## Data

In [2]:
df = load_training_data(ROOT)
test = df[df["split"] == "test"]
own = load_own_testsets(ROOT)
cal = load_calibration_no(ROOT)
print(f"offentlig test: {len(test):,}   egne testsett: {len(own):,}   kalibrering: {len(cal):,}")

offentlig test: 20,618   egne testsett: 1,192   kalibrering: 398


## Score alle tekstene

Prompt Guard 2 har to klasser, og klasse 1 er angrep. Tekster over 512 tokens kuttes, slik modellen er trent. Scorene lagres per tekst, så ingenting regnes ut to ganger.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).float().eval()
print(model.config.id2label)

def key(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

cache = pd.read_parquet(CACHE) if CACHE.exists() else pd.DataFrame(columns=["key", "score"])
known = dict(zip(cache["key"], cache["score"]))

@torch.inference_mode()
def score(texts, batch_size=32):
    texts = list(texts)
    todo = sorted({t for t in texts if key(t) not in known}, key=len)
    for i in range(0, len(todo), batch_size):
        batch = todo[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, max_length=512, padding=True, return_tensors="pt")
        probs = model(**enc).logits.softmax(-1)[:, 1].tolist()
        known.update({key(t): p for t, p in zip(batch, probs)})
        if (i // batch_size) % 50 == 0:
            print(f"  {i + len(batch):,} / {len(todo):,}")
    pd.DataFrame({"key": list(known), "score": list(known.values())}).to_parquet(CACHE, index=False)
    return np.array([known[key(t)] for t in texts])

own_scores = score(own["text"])
cal_scores = score(cal["text"])
test_scores = score(test["text"])

c:\Users\muhay\Projects\prompt-injection-detector\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{0: 'LABEL_0', 1: 'LABEL_1'}


  32 / 1,188


  32 / 398


  32 / 20,617


... (8 outputs left out)

  14,432 / 20,617


  16,032 / 20,617


  17,632 / 20,617


  19,232 / 20,617


## Terskel

Samme regel som for de andre modellene: høyst 2 % av de vanlige meldingene i det norske kalibreringssettet blir blokkert. Standardterskelen (0,5) vises også, til sammenligning.

In [4]:
threshold = threshold_for_fpr(cal["label"], cal_scores, max_fpr=0.02)
print(f"Prompt Guard 2-terskel fra kalibreringssettet: {threshold:.4f} (standard er 0,5)")

Prompt Guard 2-terskel fra kalibreringssettet: 0.9797 (standard er 0,5)


## Egne testsett, per språk

In [5]:
by_lang = own.assign(source=own["language"])
pd.concat({
    "kalibrert terskel": evaluate_by_source(by_lang, own_scores, threshold),
    "terskel 0,5": evaluate_by_source(by_lang, own_scores, 0.5),
})[["precision", "recall", "fpr", "roc_auc"]]

precision  recall    fpr  roc_auc
                  kilde                                   
kalibrert terskel en         0.956   0.280  0.008    0.941
                  no         0.941   0.207  0.008    0.948
terskel 0,5       en         0.929   0.341  0.016    0.941
                  no         0.959   0.302  0.008    0.948

Sammenligningen med baselinen, ProtectAI og mDeBERTa, og feilanalysen, ligger i notebook 06.